# Linear Layer

## Summary

The Linear TransformationIn the context of modern neural networks, a linear layer (also known as a fully connected or dense layer) is typically represented as:

$$y = xW^T + b$$

**Component Breakdown**

* $x$: The input tensor. In a Transformer, this is usually a vector of shape `[batch_size, sequence_length, input_features]`.
* $W^T$: The transpose of the weight matrix.
* $b$: The bias vector (optional), which is added to the result of the matrix multiplication.
* $y$: The output tensor.
<!-- $$
\displaystyle
y = x W^T
$$ -->

### Implimention

In [2]:
import torch
import math
from torch import Tensor
from jaxtyping import Float
import torch.nn as nn
from torch.nn.parameter import Parameter

class Linear(nn.Module):
    __constants__ = ["in_features", "out_features"]
    
    in_features: int
    out_features: int
    weight: Tensor

    def __init__(
        self,
        in_features: int,                   # input dimensions
        out_features: int,                  # output dimensions
        device: torch.device | None = None, # CPU or GPU
        dtype: torch.dtype | None = None    # float32, 64, etc
    ) -> None:
        
        factory_kwargs = {"device": device, "dtype": dtype}
        
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.weight = Parameter(
            torch.empty((out_features, in_features), **factory_kwargs)
        )
        self.reset_parameters()
    
    def reset_parameters(self) -> None:
        nn.init.trunc_normal_(self.weight)
    
    def forward(self, x: torch.Tensor) -> Tensor:
        return x @ self.weight.T

## Step-by-Step

#### Definition `Linear` class

In [1]:
import torch
import math
from torch import Tensor
from jaxtyping import Float
import torch.nn as nn
from torch.nn.parameter import Parameter

# Inherit super class constructor `nn.Module`
class Linear(nn.Module):

    # these are fixed configuration values of the layer
    __constants__ = ["in_features", "out_features"]
    
    in_features: int
    out_features: int
    weight: Tensor
    # weight is a PyTorch Tensor and stores the weight matrix

    def __init__(
        self,
        in_features: int,                   # input dimensions
        out_features: int,                  # output dimensions
        device: torch.device | None = None, # CPU or GPU
        dtype: torch.dtype | None = None    # float32, 64, etc
    ) -> None:
        
        # dictionary is later passed into torch.empty() **operator means: it unpacks the dictionary into keyword arguments
        factory_kwargs = {"device": device, "dtype": dtype}
        
        # inherit parent class constructor
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        # Creating the Weight Matrix
        # If you don't wrap it in Parameter, the optimizer won't update it. so it means trainable weight matrix
        self.weight = Parameter(
            torch.empty((out_features, in_features), **factory_kwargs)
        )
        
        self.reset_parameters()


### Initialization

**`trunc_normal_`** initialization

![Truncated Normal Distribution](../images/TruncatedNormal.png)

Fill the input Tensor with values drawn from a truncated normal distribution.

The values are effectively drawn from the normal distribution $ \mathcal{N}(\textrm{mean},\textrm{std}^2)$ with values outside $[a,b]$ redrawn until they are within the bounds. 

The method used for generating the random values works best when $a≤\textrm{mean}≤b$.

In [ ]:
def reset_parameters(self) -> None:
    nn.init.trunc_normal_(self.weight)

### Matrix Multiplation

In [ ]:
def forward(self, x: torch.Tensor) -> Tensor:
    return x @ self.weight.T

#### Appendix

**PyTorch Weight Storage & Matrix Multiplication**

In PyTorch, linear layers typically store weights in the shape `(out_features, in_features)`. 
While it might seem counterintuitive compared to the standard mathematical notation $y = W x$, <br>
there are significant hardware advantages to this approach.

> **Note:** Even if you stored the weight as `(in_features, out_features)`, the operation `x @ self.weight` would technically work, but it wouldn't benefit from the same level of low-level optimization.


**Memory Layout and GEMM**

* **Storage Shape:** `(out_features, in_features)` row-major form
* **Execution:** When you run `x @ weight.T` which is row-form major, PyTorch leverages highly optimized routines that maximize cache hits.

---

**Memory Layout: Row-Major vs. Column-Major**

The way a language stores a 2D array in linear memory (RAM) dictates how fast certain operations will be.

![Row-Major vs. Column-Major](../images/row-major-column-major.webp)


**Comparison Table**

| Feature | Row-Major Order | Column-Major Order |
| --- | --- | --- |
| **Storage Logic** | Elements are stored row-by-row. | Elements are stored column-by-column. |
| **Adjacent Elements** |  and  are neighbors. |  and  are neighbors. |
| **Performance** | Faster row-wise access. | Faster column-wise access. |
| **Primary Use** | General purpose programming. | Scientific & Mathematical computing. |

**Language Ecosystems**

| Type | Key Languages |
| --- | --- |
| **Row-Major** (C-Style) | C, C++, Python (NumPy default), Pascal, SAS, HLSL |
| **Column-Major** (Fortran-Style) | Fortran, MATLAB, R, Julia, Scilab, OpenGL (GLSL) |

### Example

In [3]:
# 1. Setup dimensions
batch_size = 4
in_features = 8
out_features = 4

# 2. Instantiate your custom layer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Linear(in_features, out_features, device=device)

# 3. Create a dummy input tensor
# Shape: (Batch Size, In Features)
input_tensor: Float[Tensor, "batch in_features"] = torch.randn(batch_size, in_features).to(device)

# 4. Run the forward pass
output = model(input_tensor)

# 5. Check the results
print(f"Input Shape:  {input_tensor.shape}")
print("Input Tensor:")
print(input_tensor)
print(f"\nWeight Shape: {model.weight.shape}")
print("Weight Tensor:")
print(model.weight)
print(f"\nOutput Shape: {output.shape}")
print("Output Tensor:")
print(output)

Input Shape:  torch.Size([4, 8])
Input Tensor:
tensor([[ 1.7361e+00,  5.4547e-01,  3.3925e-01, -5.4480e-01,  3.1430e-04,
         -4.3488e-03, -1.1595e+00,  8.3551e-01],
        [ 4.8148e-01,  1.2996e+00,  7.5173e-01, -2.1811e+00,  3.0995e+00,
         -5.2967e-01, -2.3623e+00,  3.1557e-01],
        [-1.7284e+00, -8.1833e-01,  7.3731e-01,  9.2427e-01,  9.7703e-01,
         -5.3740e-01,  1.6447e+00,  2.1382e-01],
        [ 4.0112e-01, -1.7778e+00,  1.6594e+00,  2.2391e-01,  6.4900e-01,
         -1.5424e+00,  2.6902e-02, -7.9182e-05]])

Weight Shape: torch.Size([4, 8])
Weight Tensor:
Parameter containing:
tensor([[ 0.0026,  0.7479,  0.5225, -0.0358, -0.8115,  1.4519, -0.2072,  0.5889],
        [ 0.7853, -0.6534,  0.2480, -0.2700,  0.0939, -1.1037,  1.8233, -0.9891],
        [ 0.8080, -0.2077,  1.0171, -0.0443, -1.0132, -0.2487, -0.1482,  1.0280],
        [ 0.3975, -0.1325,  0.0816,  1.1986, -1.6645,  0.5661, -0.9332,  1.6324]],
       requires_grad=True)

Output Shape: torch.Size([4, 4])